# DeepOMAPNet — Training Notebook

End-to-end workflow: generate synthetic CITE-seq data → build graphs → train → evaluate.

In [ ]:
import sys, os, importlib, warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

## 1. Load Data (Synthetic CITE-seq)

In [ ]:
from scripts.data_provider.synthetic_citeseq import generate_citeseq_dataset, CELL_TYPE_NAMES

# Generate synthetic CITE-seq dataset (replace with real data loader if available)
dataset = generate_citeseq_dataset(n_normal=1500, n_aml=1500, seed=42)

print(f'Total cells  : {dataset.n_cells}')
print(f'RNA features : {dataset.n_genes}')
print(f'ADT proteins : {dataset.n_adts}')
print(f'AML cells    : {dataset.aml_label.sum()} ({dataset.aml_label.mean()*100:.0f}%)')
print()
print('Cell-type breakdown:')
for i, name in enumerate(CELL_TYPE_NAMES):
    count = (dataset.celltype_label == i).sum()
    print(f'  {name:<12}: {count:4d} cells ({count/dataset.n_cells*100:.1f}%)')

# Load adata_gene_train.h5ad from /projects/vanaja_lab/satya/Datasets

In [ ]:
import anndata as ad
import numpy as np
from sklearn.model_selection import train_test_split

# ── 1. Load all datasets ──────────────────────────────────────────────────────
base_path = "/projects/vanaja_lab/satya/Datasets"

files = {
    "GSMControlRNA": "GSMControlRNA.h5ad",
    "ControlADT":    "ControlADT.h5ad",
    "AMLARNA":       "AMLARNA.h5ad",
    "AMLAADT":       "AMLAADT.h5ad",
    "AMLBRNA":       "AMLBRNA.h5ad",
    "AMLBADT":       "AMLBADT.h5ad",
}

adatas = []
for label, fname in files.items():
    adata = ad.read_h5ad(f"{base_path}/{fname}")
    adata.obs["source"] = label          # tag each cell with its origin file
    adatas.append(adata)
    print(f"Loaded {label}: {adata.shape}")

# ── 2. Combine all datasets ───────────────────────────────────────────────────
combined = ad.concat(adatas, join="outer", label="source", keys=list(files.keys()))
combined.obs_names_make_unique()
print(f"\nCombined shape: {combined.shape}")  # (total_cells, total_features)

# ── 3. Split indices: 80% train / 10% val / 10% test ─────────────────────────
indices = np.arange(combined.n_obs)

train_idx, temp_idx = train_test_split(indices, test_size=0.20, random_state=42)
val_idx,   test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

# ── 4. Slice into split AnnData objects ───────────────────────────────────────
train_data = combined[train_idx].copy()
val_data   = combined[val_idx].copy()
test_data  = combined[test_idx].copy()

print(f"\nTrain : {train_data.shape}")
print(f"Val   : {val_data.shape}")
print(f"Test  : {test_data.shape}")

# ── 5. (Optional) Save the splits ─────────────────────────────────────────────
train_data.write_h5ad(f"{base_path}/train.h5ad")
val_data.write_h5ad(f"{base_path}/val.h5ad")
test_data.write_h5ad(f"{base_path}/test.h5ad")
print("\nSplits saved.")

## 2. Train / Test Split → AnnData Objects

In [ ]:
import anndata
import numpy as np
from scipy import sparse

# ── Helper to extract RNA matrix + labels from an AnnData split ──────────────
def make_split(adata, tag):
    # RNA matrix — use raw counts from layers if available, else .X
    X = adata.layers['counts'] if 'counts' in adata.layers else adata.X
    rna = anndata.AnnData(X=sparse.csr_matrix(X.astype('float32')))
    rna.obs_names = adata.obs_names.tolist()
    rna.var_names = adata.var_names.tolist()

    # Copy over all obs metadata
    rna.obs = adata.obs.copy()

    # Copy embeddings
    for key in adata.obsm:
        rna.obsm[key] = adata.obsm[key]

    # Extract labels from obs
    aml_labels      = adata.obs['aml'].values
    celltype_labels = adata.obs['Cell_type_identity'].values

    print(f"{tag}: {rna.n_obs} cells  |  {rna.n_vars} genes")
    return rna, aml_labels, celltype_labels

# ── Build all three splits ────────────────────────────────────────────────────
rna_adata, aml_labels_train, celltype_labels_train = make_split(train_data, "Train")
rna_val,   aml_labels_val,   celltype_labels_val   = make_split(val_data,   "Val  ")
rna_test,  aml_labels_test,  celltype_labels_test  = make_split(test_data,  "Test ")

## 3. Build PyTorch Geometric Graphs

In [ ]:
from scripts.data_provider.graph_data_builder import build_pyg_data

print('Building RNA graph (train)...')
rna_pyg = build_pyg_data(rna_adata)
print(f'  {rna_pyg}')

print('Building ADT graph (train)...')
adt_pyg = build_pyg_data(adt_adata)
print(f'  {adt_pyg}')

assert rna_pyg.num_nodes == adt_pyg.num_nodes, 'Node count mismatch!'
print('Graphs built successfully.')

## 4. Train the Model

In [ ]:
from scripts.trainer.gat_trainer import train_gat_transformer_fusion

num_cell_types = len(CELL_TYPE_NAMES)  # 7

training_config = dict(
    epochs=100,
    seed=42,
    learning_rate=1e-3,
    weight_decay=1e-4,
    dropout_rate=0.4,
    hidden_channels=64,
    num_heads=4,
    num_attention_heads=4,
    num_layers=2,
    use_mixed_precision=torch.cuda.is_available(),
    early_stopping_patience=10,
    num_cell_types=num_cell_types,
)

(
    trained_model,
    rna_data_with_masks,
    adt_data_with_masks,
    training_history,
    adt_mean, adt_std,
    node_degrees_rna, node_degrees_adt,
    clustering_coeffs_rna, clustering_coeffs_adt,
) = train_gat_transformer_fusion(
    rna_data=rna_pyg,
    adt_data=adt_pyg,
    aml_labels=aml_labels_train,
    rna_anndata=rna_adata,
    adt_anndata=adt_adata,
    celltype_labels=celltype_labels_train,
    celltype_weight=0.5,
    **training_config,
)

print(f"\nBest val  R²: {max(training_history['val_R2']):.4f}")
print(f"Final test R²: {training_history['test_R2'][-1]:.4f}")

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(training_history['train_loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(training_history['val_MSE'],  label='Val')
axes[1].plot(training_history['test_MSE'], label='Test')
axes[1].set_title('ADT MSE'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(training_history['val_R2'],  label='Val')
axes[2].plot(training_history['test_R2'], label='Test')
axes[2].set_title('ADT R²'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Evaluate on Held-Out Test Set

In [ ]:
import scipy.sparse as sp
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, roc_auc_score

device = next(trained_model.parameters()).device

# Build test graph
print('Building test graph...')
rna_test_pyg = build_pyg_data(rna_test)

# Prepare features
X_test = rna_test.X.toarray() if sp.issparse(rna_test.X) else np.array(rna_test.X)
rna_features  = torch.tensor(X_test, dtype=torch.float32).to(device)
edge_index_rna = rna_test_pyg.edge_index.to(device)

# Inference
trained_model.eval()
with torch.no_grad():
    adt_pred, aml_pred, fused_emb = trained_model(
        x=rna_features,
        edge_index_rna=edge_index_rna,
        edge_index_adt=None,
    )

adt_pred_np = adt_pred.cpu().numpy()
adt_true_np = adt_test.X if not sp.issparse(adt_test.X) else adt_test.X.toarray()

# ADT regression metrics
r2  = r2_score(adt_true_np, adt_pred_np)
mse = mean_squared_error(adt_true_np, adt_pred_np)

# AML classification metrics
aml_probs  = torch.sigmoid(aml_pred).cpu().numpy().flatten()
aml_binary = (aml_probs > 0.5).astype(int)
acc = accuracy_score(aml_labels_test, aml_binary)
auc = roc_auc_score(aml_labels_test, aml_probs)

print('=== Held-out Test Results ===')
print(f'ADT  R²      : {r2:.4f}')
print(f'ADT  MSE     : {mse:.4f}')
print(f'AML  Accuracy: {acc:.4f}')
print(f'AML  AUC-ROC : {auc:.4f}')

## 7. Per-Protein Correlation (Test Set)

In [ ]:
from scipy.stats import pearsonr

n_proteins = adt_pred_np.shape[1]
protein_names = dataset.adt_names
correlations = [pearsonr(adt_true_np[:, p], adt_pred_np[:, p])[0] for p in range(n_proteins)]

# Bar chart
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(protein_names, correlations)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('Pearson r')
ax.set_title('Per-Protein Prediction Correlation (Test Set)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

print(f'Mean correlation: {np.mean(correlations):.4f}')
print(f'Best protein   : {protein_names[int(np.argmax(correlations))]} ({max(correlations):.4f})')

## 8. Save Model

In [ ]:
torch.save(trained_model.state_dict(), 'DeepOMAPNet_weights.pth')
print('Model saved to DeepOMAPNet_weights.pth')